# 05 — Hyperparameter Tuning (XGBoost, LightGBM, CatBoost)

`03_model_comparison.ipynb` showed every model at default settings, and flagged the boosting
models (XGBoost especially) as underperforming their potential — large train/val gaps, a textbook
overfitting signature. Rather than assume which model to present in advance, this notebook tunes
**all three major gradient-boosting options** the same rigorous way, so the final model choice is
based on real, comparable evidence, not on which one the team happened to start with.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.model_selection import RandomizedSearchCV, cross_val_score
from xgboost import XGBRegressor

from src.preprocessing import get_processed_data, get_cv_splitter
from src.config import RANDOM_STATE

data = get_processed_data()
kfold = get_cv_splitter()

## Confirm the untuned baseline

Reproduces `03_model_comparison.ipynb`'s XGBoost row exactly, since this notebook uses the same
`get_processed_data()`/`get_cv_splitter()` — this is the number tuning needs to beat.

In [ ]:
baseline = XGBRegressor(random_state=RANDOM_STATE)
cv_scores = cross_val_score(baseline, data.X_train, data.y_train, cv=kfold, scoring="r2")
baseline.fit(data.X_train, data.y_train)

val_r2 = r2_score(data.y_val, baseline.predict(data.X_val))
train_r2 = r2_score(data.y_train, baseline.predict(data.X_train))

print(f"cv_r2_mean={cv_scores.mean():.4f}  cv_r2_std={cv_scores.std():.4f}  "
      f"val_r2={val_r2:.4f}  train_r2={train_r2:.4f}  gap={train_r2-val_r2:.4f}")

## Randomized search over regularization-focused hyperparameters

Every parameter here controls how much a tree is *allowed* to fit the training data —
tuning is deliberately about reining the model in, not making it more powerful:

- `max_depth` — how many yes/no questions deep a single tree can go (shallower = simpler = less overfitting)
- `min_child_weight` — how much evidence a split needs before it's allowed to happen
- `subsample` / `colsample_bytree` — train each tree on a random subset of rows/columns, so no single tree can memorize everything
- `reg_alpha` / `reg_lambda` — L1/L2 regularization, a direct penalty for complexity
- `learning_rate` + `n_estimators` — smaller steps, more of them, generally generalizes better than a few large steps

`RandomizedSearchCV` tries a random sample of combinations (cheaper than testing every single
combination) and picks the one that scores best across the same 5-fold `kfold` used everywhere
else in this repo.

In [ ]:
param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [2, 3, 4, 5, 6],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "min_child_weight": [1, 3, 5, 10, 20],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.5, 0.6, 0.7, 0.8, 1.0],
    "reg_alpha": [0, 0.1, 1, 5],
    "reg_lambda": [1, 5, 10, 20],
}

search = RandomizedSearchCV(
    XGBRegressor(random_state=RANDOM_STATE),
    param_distributions=param_dist,
    n_iter=60,
    scoring="r2",
    cv=kfold,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
search.fit(data.X_train, data.y_train)

print("Best CV R2:", round(search.best_score_, 4))
print("Best params:", search.best_params_)

## Final, untouched check against `X_val`

Per the README's own warning: repeatedly searching against the same 5 CV folds risks quietly
overfitting the *hyperparameters* to those folds. `X_val` was never touched during the search
above, so this is an honest, independent check of the tuned model.

In [ ]:
best_model = search.best_estimator_

val_r2_tuned = r2_score(data.y_val, best_model.predict(data.X_val))
train_r2_tuned = r2_score(data.y_train, best_model.predict(data.X_train))
rmse_tuned = root_mean_squared_error(data.y_val, best_model.predict(data.X_val))

print(f"cv_r2_mean={search.best_score_:.4f}  val_r2={val_r2_tuned:.4f}  "
      f"train_r2={train_r2_tuned:.4f}  gap={train_r2_tuned-val_r2_tuned:.4f}  val_rmse={rmse_tuned:.4f}")

## XGBoost: results

| | cv_r2_mean | val_r2 | train_r2 | gap | val_rmse |
|---|---|---|---|---|---|
| XGBoost, default params | 0.5035 | 0.4801 | 0.8839 | 0.4038 | — |
| **XGBoost, tuned** | **0.6001** | **0.5780** | 0.6242 | **0.0463** | 8.14 |

Best params: `max_depth=2, min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
learning_rate=0.05, n_estimators=200, reg_lambda=5, reg_alpha=0`.

## LightGBM: same search, same rules

Same idea as the XGBoost search above — restraint-focused parameters only (`num_leaves`,
`max_depth`, `min_child_samples`, `subsample`, `colsample_bytree`, `reg_alpha`, `reg_lambda`,
`learning_rate`) — scored on the identical shared `kfold`, so the result is directly comparable to
XGBoost's.

In [ ]:
from lightgbm import LGBMRegressor

lgbm_baseline = LGBMRegressor(random_state=RANDOM_STATE, verbosity=-1)
lgbm_cv = cross_val_score(lgbm_baseline, data.X_train, data.y_train, cv=kfold, scoring="r2")
lgbm_baseline.fit(data.X_train, data.y_train)
lgbm_base_val = r2_score(data.y_val, lgbm_baseline.predict(data.X_val))
lgbm_base_train = r2_score(data.y_train, lgbm_baseline.predict(data.X_train))
print(f"LightGBM default: cv={lgbm_cv.mean():.4f} val={lgbm_base_val:.4f} "
      f"train={lgbm_base_train:.4f} gap={lgbm_base_train-lgbm_base_val:.4f}")

lgbm_param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "num_leaves": [7, 15, 31, 63],
    "max_depth": [-1, 3, 4, 5, 6],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "min_child_samples": [5, 10, 20, 40, 60],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.5, 0.6, 0.7, 0.8, 1.0],
    "reg_alpha": [0, 0.1, 1, 5],
    "reg_lambda": [1, 5, 10, 20],
}
lgbm_search = RandomizedSearchCV(LGBMRegressor(random_state=RANDOM_STATE, verbosity=-1),
                                  param_distributions=lgbm_param_dist, n_iter=60, scoring="r2",
                                  cv=kfold, random_state=RANDOM_STATE, n_jobs=-1)
lgbm_search.fit(data.X_train, data.y_train)
lgbm_best = lgbm_search.best_estimator_
lgbm_val = r2_score(data.y_val, lgbm_best.predict(data.X_val))
lgbm_train = r2_score(data.y_train, lgbm_best.predict(data.X_train))
lgbm_rmse = root_mean_squared_error(data.y_val, lgbm_best.predict(data.X_val))
print(f"LightGBM tuned:   cv={lgbm_search.best_score_:.4f} val={lgbm_val:.4f} "
      f"train={lgbm_train:.4f} gap={lgbm_train-lgbm_val:.4f} rmse={lgbm_rmse:.4f}")
print("best params:", lgbm_search.best_params_)

**LightGBM results:**

| | cv_r2_mean | val_r2 | train_r2 | gap | val_rmse |
|---|---|---|---|---|---|
| LightGBM, default params | 0.5694 | 0.5484 | 0.7701 | 0.2217 | — |
| **LightGBM, tuned** | **0.6006** | **0.5764** | 0.6288 | **0.0524** | 8.15 |

Best params: `subsample=0.9, reg_lambda=1, reg_alpha=5, num_leaves=7, n_estimators=500,
min_child_samples=10, max_depth=5, learning_rate=0.01, colsample_bytree=0.6`.

## CatBoost: same search, same rules

Same restraint-focused approach, using CatBoost's equivalent parameters (`depth`, `l2_leaf_reg`,
`min_data_in_leaf`, `learning_rate`, `iterations`). Search space kept smaller (20 combinations
instead of 60) since CatBoost trains noticeably slower than the other two — a real practical cost,
not just an accuracy question, worth remembering when comparing the three.

In [ ]:
from catboost import CatBoostRegressor

cb_baseline = CatBoostRegressor(random_state=RANDOM_STATE, verbose=False)
cb_cv = cross_val_score(cb_baseline, data.X_train, data.y_train, cv=kfold, scoring="r2")
cb_baseline.fit(data.X_train, data.y_train)
cb_base_val = r2_score(data.y_val, cb_baseline.predict(data.X_val))
cb_base_train = r2_score(data.y_train, cb_baseline.predict(data.X_train))
print(f"CatBoost default: cv={cb_cv.mean():.4f} val={cb_base_val:.4f} "
      f"train={cb_base_train:.4f} gap={cb_base_train-cb_base_val:.4f}")

cb_param_dist = {
    "iterations": [200, 300],
    "depth": [2, 3, 4, 5],
    "learning_rate": [0.03, 0.05, 0.1],
    "l2_leaf_reg": [1, 3, 5, 10, 20],
    "min_data_in_leaf": [1, 5, 10, 20],
}
cb_search = RandomizedSearchCV(CatBoostRegressor(random_state=RANDOM_STATE, verbose=False),
                                param_distributions=cb_param_dist, n_iter=20, scoring="r2",
                                cv=kfold, random_state=RANDOM_STATE, n_jobs=4)
cb_search.fit(data.X_train, data.y_train)
cb_best = cb_search.best_estimator_
cb_val = r2_score(data.y_val, cb_best.predict(data.X_val))
cb_train = r2_score(data.y_train, cb_best.predict(data.X_train))
cb_rmse = root_mean_squared_error(data.y_val, cb_best.predict(data.X_val))
print(f"CatBoost tuned:   cv={cb_search.best_score_:.4f} val={cb_val:.4f} "
      f"train={cb_train:.4f} gap={cb_train-cb_val:.4f} rmse={cb_rmse:.4f}")
print("best params:", cb_search.best_params_)

**CatBoost results:**

| | cv_r2_mean | val_r2 | train_r2 | gap | val_rmse |
|---|---|---|---|---|---|
| CatBoost, default params | 0.5728 | 0.5332 | 0.7852 | 0.2520 | — |
| **CatBoost, tuned** | **0.5955** | **0.5680** | 0.6364 | **0.0684** | 8.23 |

Best params: `min_data_in_leaf=5, learning_rate=0.1, l2_leaf_reg=5, iterations=300, depth=2`.

## Final three-way comparison and recommendation

| Model | Tuned val_r2 (decision metric) | Tuned gap | Tuned val_rmse |
|---|---|---|---|
| **XGBoost** | **0.578** | **0.046** | 8.14 |
| LightGBM | 0.576 | 0.052 | 8.15 |
| CatBoost | 0.568 | 0.068 | 8.23 |

**Once properly tuned, all three are essentially tied** — the gap between XGBoost's 0.578 and
LightGBM's 0.576 is well within the fold-to-fold noise seen elsewhere in this project (CV standard
deviations around 0.03-0.04), not a meaningful difference. CatBoost trails both slightly on every
metric here *and* took roughly 5x longer to tune for that result — a real practical cost.

**Recommendation: XGBoost**, primarily on practical grounds rather than a decisive accuracy edge:
it's the model the team started with, it's named explicitly in the course brief, it has the
smallest overfitting gap of the three (most trustworthy/reproducible), and it has the largest
documentation/community base. LightGBM is an equally defensible backup if the team prefers it.
`val_r2` (performance on data the model never trained or was tuned against) is used as the primary
decision metric here rather than `cv_r2_mean` or `train_r2`, since it's the closest available proxy
to real-world generalization short of an actual Kaggle submission.